In [ ]:
import os
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv
from tqdm import tqdm
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from bert_score import score as bert_score
import numpy as np
import seaborn as sns
from scipy.stats import ttest_rel
import matplotlib.pyplot as plt
# Load Toxic-BERT model and tokenizer
model_name = "unitary/toxic-bert"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

In [ ]:
def process_transcripts(folder_path):
    """
    Process text files in a folder and read their content into a DataFrame.
    
    Parameters:
    folder_path (str): Path to the directory containing the text files.
    
    Returns:
    pd.DataFrame: DataFrame containing the file names and transcripts.
    """
    # Initialize an empty list to store the file data
    data = []

    # Iterate over all files in the folder
    for file_name in os.listdir(folder_path):
        # Ensure the file is a text file
        if file_name.endswith(".txt"):
            file_path = os.path.join(folder_path, file_name)
            
            # Read the content of the text file
            with open(file_path, "r", encoding="utf-8") as file:
                text = file.read()
            
            # Append the data as a tuple (file name, transcript content)
            data.append((file_name, text))
    
    # Create a DataFrame from the collected data
    df = pd.DataFrame(data, columns=["file_name", "transcript"])
    
    return df

In [ ]:
folder_path = 
df = process_transcripts(folder_path)

df.head(5)

# Generate Medical Note

In [ ]:
load_dotenv()

client = OpenAI(
    api_key=)

In [ ]:
def generate_medical_notes(dataframe, column_name, num_simulations=5):
    """
    Generate SOAP medical notes multiple times and append each simulation as a new column.

    Args:
        dataframe (pd.DataFrame): The input DataFrame.
        column_name (str): The column containing transcripts to process.
        num_simulations (int): Number of simulations to perform.

    Returns:
        pd.DataFrame: A DataFrame with additional columns for each simulation.
    """
    for i in range(1, num_simulations + 1):
        new_column_name = f"{column_name}_simulated_notes_{i}"
        notes = []

        for transcript in tqdm(dataframe[column_name], desc=f"Generating Simulation {i}"):
            try:
                # Prepare the additional instructions
                additional_instructions = """
                Summarize the transcript by accurately classifying information into each SOAP section:
                Subjective: Include the patient's description of symptoms, medical history, and any concerns raised during the conversation.
                Objective: Include the physician's observations, any diagnostic test results, or physical examination findings.
                Assessment: Provide the physician's diagnosis or clinical impression.
                Plan: Outline the treatment plan, recommended next steps, and any follow-up actions discussed.
                Ensure the summary is clear, uses appropriate medical terminology, and is structured to facilitate integration into the patient's medical record.
                """

                # Create prompt for the model
                user_content = transcript + '\n\n' + additional_instructions

                # Make the API call
                chat_completion = client.chat.completions.create(
                    model="gpt-4o",
                    messages=[
                        {
                            "role": "system",
                            "content": "You are an AI assistant tasked with creating a SOAP note based on the following transcript."
                        },
                        {
                            "role": "user",
                            "content": user_content
                        }
                    ]
                )

                # Extract the response content
                response_content = chat_completion.choices[0].message.content
                notes.append(response_content)
            except Exception as e:
                # Append error message for rows that fail
                notes.append(f"Error: {str(e)}")

        # Add the new column to the DataFrame
        dataframe[new_column_name] = notes

    return dataframe

In [ ]:
df.rename(columns={
    'transcript_simulated_notes_1': 'gpt_note_1',
    'transcript_simulated_notes_2': 'gpt_note_2',
    'transcript_simulated_notes_3': 'gpt_note_3',
    'transcript_simulated_notes_4': 'gpt_note_4',
    'transcript_simulated_notes_5': 'gpt_note_5'
}, inplace=True)

# Read in reference text

In [ ]:
def read_reference_texts(reference_dir):
    """
    Reads text files in a directory and creates a DataFrame with two columns:
    - 'file_id': first 16 characters of each file name
    - 'content': content of the corresponding file

    Parameters:
        reference_dir (str): The directory containing reference text files.

    Returns:
        pd.DataFrame: DataFrame with file identifiers and file contents.
    """
    data = []

    # Loop through files in the directory
    for file_name in os.listdir(reference_dir):
        file_path = os.path.join(reference_dir, file_name)
        # Check if the path is a file
        if os.path.isfile(file_path):
            try:
                # Extract the first 16 characters of the file name
                file_id = file_name[:16]
                # Read the content of the file with multiple encodings fallback
                try:
                    with open(file_path, 'r', encoding='utf-8') as file:
                        content = file.read()
                except UnicodeDecodeError:
                    with open(file_path, 'r', encoding='cp1252') as file:  # Common fallback encoding
                        content = file.read()
                # Append to data list
                data.append({'file_name': file_id, 'reference_text': content})
            except Exception as e:
                print(f"Error reading {file_name}: {e}")

    # Create DataFrame
    df = pd.DataFrame(data)
    return df

In [ ]:
# Example usage
reference_dir = 
reference_df = read_reference_texts(reference_dir)
reference_df.head(10)

In [ ]:
df['file_name'] = df['file_name'].str[:16]
df.head(5)

In [ ]:
df_with_reference = pd.merge(reference_df, df, on='file_name', how='inner')
df_with_reference.shape

# Save the GPT note

In [ ]:
def save_gpt_notes(file_path, dataframe):
    """
    Save GPT notes from a dataframe into folders, with each folder corresponding to a GPT note column.

    Parameters:
    file_path (str): The base directory where folders will be created.
    dataframe (pd.DataFrame): The dataframe containing the columns 'file_name', 'gpt_note_1' to 'gpt_note_5'.
    """
    # Ensure the base directory exists
    if not os.path.exists(file_path):
        os.makedirs(file_path)

    # Loop through GPT note columns
    for i in range(1, 6):
        folder_name = os.path.join(file_path, str(i))
        os.makedirs(folder_name, exist_ok=True)

        note_column = f"gpt_note_{i}"

        if note_column in dataframe.columns:
            for _, row in dataframe.iterrows():
                file_name = row['file_name']
                note_content = row[note_column]

                if pd.notnull(note_content):  # Check if note content is not NaN
                    note_file_path = os.path.join(folder_name, f"{file_name}_gpt_note_{i}.txt")

                    with open(note_file_path, 'w', encoding='utf-8') as note_file:
                        note_file.write(str(note_content))


# Read in Llama notes

In [ ]:
# Path to the main folder containing subfolders 1 to 5
main_folder = 
# Initialize an empty DataFrame
columns = ['file_name', 'llama_note_1', 'llama_note_2', 'llama_note_3', 'llama_note_4', 'llama_note_5']
notes_df = pd.DataFrame(columns=columns)

# Loop through subfolders 1 to 5
for i in range(1, 6):
    subfolder = os.path.join(main_folder, str(i))
    if os.path.exists(subfolder):
        for file in os.listdir(subfolder):
            if file.endswith('.txt'):
                file_path = os.path.join(subfolder, file)
                with open(file_path, 'r', encoding='utf-8') as f:
                    content = f.read()
                
                # Extract the file name (first 16 characters of the file)
                file_name = file[:16]

                # Find the row in the DataFrame with this file name or create a new one
                if file_name not in notes_df['file_name'].values:
                    new_row = {'file_name': file_name, f'llama_note_{i}': content}
                    notes_df = pd.concat([notes_df, pd.DataFrame([new_row])], ignore_index=True)
                else:
                    notes_df.loc[notes_df['file_name'] == file_name, f'llama_note_{i}'] = content

# Display the DataFrame
notes_df.reset_index(drop=True, inplace=True)
print(notes_df)


In [ ]:
direct_evaluation_repeated_df = pd.merge(notes_df, df_with_reference, on='file_name', how='inner')

# Display the first few rows of the resulting dataframe
direct_evaluation_repeated_df.head(10)

I am generating llama note in another notebook. Once finished, I will merege those data with data here and calculate all the scores using function below

# Calculate score

In [ ]:
# Function to calculate toxicity
def calculate_toxicity(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
    scores = torch.nn.functional.softmax(outputs.logits, dim=-1)
    return scores[0][1].item()  # Toxicity score

# Function to calculate BLEU scores
def calculate_bleu(reference, candidate, n):
    smoothing = SmoothingFunction().method4
    weights = tuple([1.0 / n] * n) + tuple([0] * (4 - n))
    return sentence_bleu([reference.split()], candidate.split(), weights=weights, smoothing_function=smoothing)

# Function to calculate BLEU cumulative scores
def calculate_bleu_cumulative(reference, candidate):
    smoothing = SmoothingFunction().method4
    bleu1 = sentence_bleu([reference.split()], candidate.split(), weights=(1, 0, 0, 0), smoothing_function=smoothing)
    bleu2 = sentence_bleu([reference.split()], candidate.split(), weights=(0.5, 0.5, 0, 0), smoothing_function=smoothing)
    bleu3 = sentence_bleu([reference.split()], candidate.split(), weights=(0.33, 0.33, 0.33, 0), smoothing_function=smoothing)
    bleu4 = sentence_bleu([reference.split()], candidate.split(), weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smoothing)
    return bleu1, bleu2, bleu3, bleu4

# Function to calculate ROUGE scores
def calculate_rouge(reference, candidate):
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rouge3', 'rougeL'], use_stemmer=True)
    scores = scorer.score(reference, candidate)
    return scores

# Main function
def calculate_text_metrics(df, reference_col, text_cols):
    results = df.copy()
    for col in text_cols:
        rouge1_list, rouge2_list, rouge3_list, rougeL_list = [], [], [], []
        bleu1_list, bleu2_list, bleu3_list, bleu4_list = [], [], [], []
        bert_precision_list, bert_recall_list, bert_f1_list = [], [], []
        meteor_list = []
        toxicity_list = []

        for ref, text in tqdm(zip(results[reference_col], results[col]), total=len(results), desc=f"Processing {col}"):
            # ROUGE
            rouge_scores = calculate_rouge(ref, text)
            rouge1_list.append(rouge_scores['rouge1'].fmeasure)
            rouge2_list.append(rouge_scores['rouge2'].fmeasure)
            rouge3_list.append(rouge_scores['rouge3'].fmeasure)
            rougeL_list.append(rouge_scores['rougeL'].fmeasure)

            # BLEU
            bleu1_list.append(calculate_bleu(ref, text, 1))
            bleu2_list.append(calculate_bleu(ref, text, 2))
            bleu3_list.append(calculate_bleu(ref, text, 3))
            bleu4_list.append(calculate_bleu(ref, text, 4))

            # BLEU Cumulative
            bleu_cum1, bleu_cum2, bleu_cum3, bleu_cum4 = calculate_bleu_cumulative(ref, text)

            # METEOR
            try:
                meteor_list.append(meteor_score([ref.split()], text.split()))
            except:
                meteor_list.append(0.0)  # Handle empty texts

            # BERTScore
            P, R, F1 = bert_score([text], [ref], lang="en", verbose=False)
            bert_precision_list.append(P.mean().item())
            bert_recall_list.append(R.mean().item())
            bert_f1_list.append(F1.mean().item())

            # Toxicity
            toxicity_list.append(calculate_toxicity(text))

        # Append results as new columns
        results[f'{col}_rouge1'] = rouge1_list
        results[f'{col}_rouge2'] = rouge2_list
        results[f'{col}_rouge3'] = rouge3_list
        results[f'{col}_rougeL'] = rougeL_list
        results[f'{col}_bleu1'] = bleu1_list
        results[f'{col}_bleu2'] = bleu2_list
        results[f'{col}_bleu3'] = bleu3_list
        results[f'{col}_bleu4'] = bleu4_list
        results[f'{col}_meteor'] = meteor_list
        results[f'{col}_bert_precision'] = bert_precision_list
        results[f'{col}_bert_recall'] = bert_recall_list
        results[f'{col}_bert_f1'] = bert_f1_list
        results[f'{col}_toxicity'] = toxicity_list

    return results


In [ ]:
notes_columns = [col for col in direct_evaluation_repeated_df.columns if 'note' in col]
# Run the `calculate_text_metrics` function for the notes columns
results_df = calculate_text_metrics(direct_evaluation_repeated_df, reference_col='reference_text', text_cols=notes_columns)

results_df.head(10)

In [ ]:
# List of base metric names 
metrics = [
    'rouge1', 'rouge2', 'rouge3', 'rougeL', 'bleu1', 'bleu2', 'bleu3',
    'bleu4', 'meteor', 'bert_precision', 'bert_recall', 'bert_f1', 'toxicity'
]

# Separate prefixes for GPT and Llama
prefixes = ['gpt', 'llama']

# Iterate over each prefix (gpt, llama)
for prefix in prefixes:
    for metric in metrics:
        # Identify the columns for this metric for the current prefix across the five notes
        metric_columns = [f'{prefix}_note_{i}_{metric}' for i in range(1, 6)]
        # Calculate the row-wise mean for this metric for the current prefix
        results_df[f'{prefix}_{metric}_average'] = results_df[metric_columns].mean(axis=1)

In [ ]:
# Define the file path
file_path = 

# Save the dataframe to a CSV file
results_df.to_csv(file_path, index=False)

print(f"File saved successfully to {file_path}")

# Data analysis Start running here!!

In [ ]:
results_df = pd.read_csv()

In [ ]:
# List of metrics to evaluate
metrics = [
    'rouge1_average', 'rouge2_average', 'rouge3_average', 'rougeL_average',
    'bleu1_average', 'bleu2_average', 'bleu3_average', 'bleu4_average',
    'meteor_average', 'bert_precision_average', 'bert_recall_average',
    'bert_f1_average', 'toxicity_average'
]

# Iterate over each metric
for metric in metrics:
    gpt_col = f'gpt_{metric}'
    llama_col = f'llama_{metric}'
    
    # Plot histograms
    plt.figure(figsize=(10, 6))
    plt.hist(results_df[gpt_col], bins=10, alpha=0.5, label='GPT', color='blue', edgecolor='black')
    plt.hist(results_df[llama_col], bins=10, alpha=0.5, label='Llama', color='orange', edgecolor='black')
    plt.title(f'Distribution of {metric}')
    plt.xlabel('Scores')
    plt.ylabel('Frequency')
    plt.legend(loc='upper right')
    plt.show()

    # Paired t-test
    t_stat, p_value = ttest_rel(results_df[gpt_col], results_df[llama_col])
    print(f"Metric: {metric}")
    print(f"Paired t-test t-statistic: {t_stat:.3f}, p-value: {p_value:.3f}")
    print("-" * 50)

## Better format

In [ ]:
metric_titles = {
    'rouge1_average': 'ROUGE-1',
    'rouge2_average': 'ROUGE-2',
    'rouge3_average': 'ROUGE-3',
    'rougeL_average': 'ROUGE-L',
    'bleu1_average': 'BLEU-1',
    'bleu2_average': 'BLEU-2',
    'bleu3_average': 'BLEU-3',
    'bleu4_average': 'BLEU-4',
    'meteor_average': 'METEOR',
    'bert_precision_average': 'BERTScore Precision',
    'bert_recall_average': 'BERTScore Recall',
    'bert_f1_average': 'BERTScore F1',
    'toxicity_average': 'Toxicity Score'
}

# Iterate over each metric
for metric in metrics:
    gpt_col = f'gpt_{metric}'
    llama_col = f'llama_{metric}'
    title = metric_titles.get(metric, metric)  # Get descriptive title or fallback to raw metric name
    
    # Plot histograms
    plt.figure(figsize=(10, 6))
    plt.hist(results_df[gpt_col], bins=10, alpha=0.5, label='GPT', color='blue', edgecolor='black')
    plt.hist(results_df[llama_col], bins=10, alpha=0.5, label='Llama', color='orange', edgecolor='black')
    plt.title(f'Distribution of {title}')
    plt.xlabel(f'{title} scores')
    plt.ylabel('Frequency')
    plt.legend(loc='upper right')
    plt.show()

    # Paired t-test
    t_stat, p_value = ttest_rel(results_df[gpt_col], results_df[llama_col])
    print(f"Metric: {title}")
    print(f"Paired t-test t-statistic: {t_stat:.3f}, p-value: {p_value:.3f}")
    print("-" * 50)

# For high resolution pictures in the paper (no title 300 dpi)

In [ ]:
metric_titles = {
    'rouge1_average': 'ROUGE-1',
    'rouge2_average': 'ROUGE-2',
    'rouge3_average': 'ROUGE-3',
    'rougeL_average': 'ROUGE-L',
    'bleu1_average': 'BLEU-1',
    'bleu2_average': 'BLEU-2',
    'bleu3_average': 'BLEU-3',
    'bleu4_average': 'BLEU-4',
    'meteor_average': 'METEOR',
    'bert_precision_average': 'BERTScore Precision',
    'bert_recall_average': 'BERTScore Recall',
    'bert_f1_average': 'BERTScore F1',
    'toxicity_average': 'Toxicity'
}

metrics_included = metrics = [
    'rouge1_average',
    'bleu1_average',
    'bert_f1_average', 
    'toxicity_average'
]

# Define the save directory
save_dir = 
os.makedirs(save_dir, exist_ok=True)  # Ensure directory exists

# Iterate over each metric
for metric in metrics_included:
    gpt_col = f'gpt_{metric}'
    llama_col = f'llama_{metric}'
    title = metric_titles.get(metric, metric)  # Get descriptive title or fallback to raw metric name
    
    # Plot histograms
    plt.figure(figsize=(10, 6))
    plt.hist(results_df[gpt_col], bins=10, alpha=0.5, label='GPT', color='blue', edgecolor='black')
    plt.hist(results_df[llama_col], bins=10, alpha=0.5, label='LLaMA', color='orange', edgecolor='black')
    plt.title(f'Distribution of {title}')
    plt.xlabel(f'{title} scores')
    plt.ylabel('Frequency')
    plt.legend(loc='upper right')

    # Save the figure at 300 DPI
    save_path = os.path.join(save_dir, f"{title.replace(' ', '_')}.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

    # Paired t-test
    t_stat, p_value = ttest_rel(results_df[gpt_col], results_df[llama_col])
    print(f"Metric: {title}")
    print(f"Paired t-test t-statistic: {t_stat:.3f}, p-value: {p_value:.3f}")
    print("-" * 50)

print(f"Figures saved successfully in: {save_dir}")

In [ ]:
keywords = [
    'rouge1_average', 'rouge2_average', 'rouge3_average', 'rougeL_average',
    'bleu1_average', 'bleu2_average', 'bleu3_average', 'bleu4_average',
    'meteor_average', 'bert_precision_average', 'bert_recall_average',
    'bert_f1_average', 'toxicity_average'
]

# Filter columns based on keywords
filtered_columns = [col for col in results_df.columns if any(keyword in col for keyword in keywords)]

# Display filtered columns
average_df = results_df[filtered_columns]
average_df.head(10)

In [ ]:
gpt_columns = [col for col in average_df.columns if col.startswith('gpt')]
gpt_mean = average_df[gpt_columns].mean()

# Select columns for LLAMA and calculate their mean
llama_columns = [col for col in average_df.columns if col.startswith('llama')]
llama_mean = average_df[llama_columns].mean()

# Combine results into a single DataFrame for clarity
average_comparison = pd.DataFrame({
    'Metric': gpt_mean.index.str.replace('gpt_', '', regex=False),
    'GPT_Mean': gpt_mean.values,
    'LLAMA_Mean': llama_mean.values
})

average_comparison.head
